In [ ]:
# Generate long synthetic ECG signal for LSTM stress model

import numpy as np
import pandas as pd
from scipy.signal.windows import gaussian

# =========================
# Parameters
# =========================
fs = 250                # sampling frequency (Hz)
duration = 960          # seconds (16 minutes)
mean_hr = 75            # bpm
hrv_strength = 0.05     # 0.02 = low HRV (stress), 0.08 = relaxed

np.random.seed(42)

# =========================
# Time axis
# =========================
t = np.arange(0, duration, 1/fs)
signal = np.zeros_like(t)

# =========================
# Generate RR intervals with variability
# =========================
mean_rr = 60 / mean_hr
rr_intervals = []

current_time = 0
while current_time < duration:
    rr = mean_rr + np.random.normal(0, hrv_strength * mean_rr)
    rr = max(0.4, rr)  # prevent unrealistic short RR
    rr_intervals.append(rr)
    current_time += rr

beat_times = np.cumsum(rr_intervals)
beat_times = beat_times[beat_times < duration]

# =========================
# QRS complex template
# =========================
qrs_width = int(0.08 * fs)  # 80ms
qrs = gaussian(qrs_width, std=qrs_width / 8)
qrs = qrs - np.min(qrs)
qrs = qrs / np.max(qrs)

# =========================
# Insert QRS complexes
# =========================
for bt in beat_times:
    idx = int(bt * fs)
    if idx + qrs_width < len(signal):
        signal[idx:idx + qrs_width] += qrs

# =========================
# Add baseline wander + noise
# =========================
baseline = 0.02 * np.sin(2 * np.pi * 0.3 * t)
noise = 0.01 * np.random.randn(len(t))

signal = signal + baseline + noise

# =========================
# Save CSV
# =========================
df = pd.DataFrame({"signal": signal})
file_path = "../../data/synthetic/long_ecg_16min.csv"
df.to_csv(file_path, index=False)

print("Saved:", file_path)
print("Duration:", duration, "seconds")
print("Use fs =", fs)

Saved: ../data/synthetic/long_ecg_16min.csv
Duration: 960 seconds
Use fs = 250
